In [0]:
%sql
use catalog end_to_end_projects;
use schema sports_equipments_sales;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

### Revenue by Product Category

In [0]:
dim_products = spark.table("dim_products")
fact_sales = spark.table("fact_sales")
display(dim_products.limit(5))
display(fact_sales.limit(5))

In [0]:
revenue_by_product_category = fact_sales.join(dim_products, on = "product_sk", how = "inner")\
    .filter(col("sale_status") == "COMPLETED")\
        .groupBy("product_category").agg(sum("final_amount").alias("total_revenue"))
display(revenue_by_product_category.limit(6))
print(f"Count of Records: {revenue_by_product_category.count()}")

In [0]:
revenue_by_product_category.write.format("delta").mode("overwrite").saveAsTable("revenue_by_product_category")

In [0]:
revenue_by_product = spark.table("revenue_by_product_category")
display(revenue_by_product)

# Revenue By Store Region

In [0]:
dim_store = spark.table("dim_store")
display(dim_store.limit(5))

In [0]:
revenue_by_store_region = fact_sales.join(dim_store, on = "store_sk", how = "inner")\
    .filter(col("sale_status") == "COMPLETED")\
        .groupBy("store_region")\
            .agg(sum("final_amount").alias("total_revenue"))
display(revenue_by_store_region)
print(f"Count Of Records: {revenue_by_store_region.count()}")

In [0]:
revenue_by_store_region.write.format("delta").mode("overwrite").saveAsTable("revenue_by_store_region")

In [0]:
revenue_by_store_region = spark.table("revenue_by_store_region")
display(revenue_by_store_region)

# Sales Count By Payment Method

In [0]:
df_completed_sales = fact_sales.filter(col("sale_status") == "COMPLETED")\
    .groupBy("payment_method")\
        .agg(count("sale_id").alias("total_sales"))
display(df_completed_sales)

In [0]:
df_completed_sales.write.format("delta").mode("overwrite").saveAsTable("sales_by_payment_method")

In [0]:
df_sales_by_payment_method = spark.table("sales_by_payment_method")
display(df_sales_by_payment_method)

In [0]:
revenue_by_product = spark.table("revenue_by_product_category")
revenue_by_region = spark.table("revenue_by_store_region")

display(revenue_by_product.agg(sum("total_revenue").alias("total_revenue_by_product")))
display(revenue_by_region.agg(sum("total_revenue").alias("total_revenue_by_region")))

In [0]:

df_sales_by_payment_method = spark.table("sales_by_payment_method")
display(df_sales_by_payment_method.agg(sum("total_sales").alias("total_sales")))